In [2]:
import pandas as pd
from datasets import load_dataset

In [3]:
print("Loading dataset from Hugging Face...")

ds = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts")
print("Available splits:", list(ds.keys()))

dfs = []
for split in ds.keys():
    temp_df = ds[split].to_pandas()
    temp_df["split_name"] = split  
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", df.shape)
df.head()

Loading dataset from Hugging Face...
Available splits: ['by_polishing', 'from_title', 'from_title_and_content']
Combined dataset shape: (8388, 6)


,original_abstract,allam_generated_abstract,jais_generated_abstract,llama_generated_abstract,openai_generated_abstract,split_name
0,كثيرا ما ارتبطت المصادر التاريخية في الأندلس خ...,يتناول هذا البحث موضوع التعليم بين النساء الأن...,تدرس هذه الدراسة دور المرأة في التعليم في الأن...,يُقدم هذا البحث دراسة شاملة حول حالة التعليم ع...,صور نظام التعليم عند المرأة الأندلسية تستند إل...,by_polishing
1,يعد العامل الثقافي احد ابرز الاسباب التي يعزى ...,يتناول هذا البحث دراسة انهيار دولة الموحدين من...,كان العامل الثقافي من بين الأسباب الرئيسية الت...,يعد العامل الثقافي أحد أبرز الأسباب التي يعزى ...,انهيار دولة الموحدين يعود بشكل كبير للعوامل ال...,by_polishing
2,شكلت تلك الجهود والمساعي الرائدة التي قام بها ...,هدفت هذه الدراسة إلى تسليط الضوء على جهود قادة...,تدرس هذه الدراسة جهود قادة الثورة الجزائرية خل...,شكلت الجهود التي بذلها قادة الثورة الجزائرية خ...,جهود قادة الثورة الجزائرية خلال المرحلة الأولى...,by_polishing
3,يقوم المقال على اشكالية الضرائب الغير شرعية في...,يتناول هذا البحث إشكالية الضرائب غير الشرعية ف...,تدرس المقالة مشكلة الضرائب غير الشرعية في مراح...,يقوم البحث على دراسة الضرائب غير الشرعية في دو...,المقال يناقش قضية الضرائب غير الشرعية في دولتي...,by_polishing
4,تتفق المصادر التاريخية المتوفرة حول موضوع تطور...,تتناول هذه الدراسة حركة الانتصار للحريات الديم...,حركة انتصار الحريات الديمقراطية (MTLD)، وهي حر...,تُظهر المصادر التاريخية المتاحة حول تطور الحرك...,حركة انتصار الحريات الديمقراطية (MTLD) في الجز...,by_polishing


In [5]:
print("Columns:\n", df.columns.tolist())

print("\nDataFrame info:")
df.info()

Columns:
 ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract', 'split_name']

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8388 entries, 0 to 8387
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   original_abstract          8388 non-null   object
 1   allam_generated_abstract   8388 non-null   object
 2   jais_generated_abstract    8388 non-null   object
 3   llama_generated_abstract   8388 non-null   object
 4   openai_generated_abstract  8388 non-null   object
 5   split_name                 8388 non-null   object
dtypes: object(6)
memory usage: 393.3+ KB


In [7]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Folders ready.")

Folders ready.


In [8]:

raw_path = "data/raw/arabic_abstracts_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8-sig")
print("Raw dataset saved to:", raw_path)

Raw dataset saved to: data/raw/arabic_abstracts_raw.csv


In [9]:
# Human abstracts
human = df[["original_abstract"]].copy()
human = human.rename(columns={"original_abstract": "text"})
human["label"] = 0  
ai_cols = [
    "allam_generated_abstract",
    "jais_generated_abstract",
    "llama_generated_abstract",
    "openai_generated_abstract"
]

ai_frames = []
for col in ai_cols:
    if col in df.columns:
        temp = df[[col]].copy()
        temp = temp.rename(columns={col: "text"})
        temp["label"] = 1  
        ai_frames.append(temp)

ai = pd.concat(ai_frames, ignore_index=True)

final_df = pd.concat([human, ai], ignore_index=True)[["text", "label"]]

print("Final binary dataset shape:", final_df.shape)
final_df.head()

Final binary dataset shape: (41940, 2)


,text,label
0,كثيرا ما ارتبطت المصادر التاريخية في الأندلس خ...,0
1,يعد العامل الثقافي احد ابرز الاسباب التي يعزى ...,0
2,شكلت تلك الجهود والمساعي الرائدة التي قام بها ...,0
3,يقوم المقال على اشكالية الضرائب الغير شرعية في...,0
4,تتفق المصادر التاريخية المتوفرة حول موضوع تطور...,0


In [10]:
phase1_path = "data/processed/phase1_dataset.csv"
final_df.to_csv(phase1_path, index=False, encoding="utf-8-sig")
print("Phase 1 dataset saved to:", phase1_path)

Phase 1 dataset saved to: data/processed/phase1_dataset.csv


In [11]:
print("Class distribution (counts):")
print(final_df["label"].value_counts())

print("\nClass distribution (proportions):")
print(final_df["label"].value_counts(normalize=True))

Class distribution (counts):
label
1    33552
0     8388
Name: count, dtype: int64

Class distribution (proportions):
label
1    0.8
0    0.2
Name: proportion, dtype: float64


In [12]:
print("Missing values per column:")
print(final_df.isnull().sum())

dup_count = final_df.duplicated(subset=["text"]).sum()
print("\nNumber of duplicate texts:", dup_count)

empty_mask = final_df["text"].astype(str).str.strip() == ""
empty_count = empty_mask.sum()
print("Number of empty texts:", empty_count)

non_arabic_mask = ~final_df["text"].astype(str).str.contains(r"[\u0600-\u06FF]", regex=True, na=False)
non_arabic_count = non_arabic_mask.sum()
print("Number of non-Arabic/mixed texts:", non_arabic_count)

word_counts = final_df["text"].astype(str).str.split().str.len()
short_mask = word_counts < 5
short_count = short_mask.sum()
print("Number of unusually short texts (< 5 words):", short_count)

Missing values per column:
text     0
label    0
dtype: int64

Number of duplicate texts: 5415
Number of empty texts: 0
Number of non-Arabic/mixed texts: 0
Number of unusually short texts (< 5 words): 0
